# Is your target data-limited? A 3-test check

You have iterated for a week. Your cross-validation has plateaued. Is the wall in **your model**, or
is it in **the data** — a ceiling no model can cross? Knowing which one you face is the single most
valuable thing you can learn mid-competition: it tells you whether to keep building or to **bank and stop**.

This notebook is a small, reusable toolkit of **three cheap, visual tests** that separate *model-limited*
from *data-limited*. It is fully self-contained (numpy + matplotlib, no data download), so you can fork it
and drop your own `(group, truth, candidate-predictions)` into the same three checks on any regression task.

The running example is this competition's geosteering target (per-well stratigraphic position `TVT`), where
the dominant error is a per-well **±15 ft "datum" ambiguity** — but everything here generalises. For the
geology behind that ambiguity, see the companion notebook linked at the end.


## A tiny synthetic stand-in

To make the tests concrete and reproducible we simulate a *competition-like* target: many **groups** (think
"wells"), each with a smooth structural trend a decent model recovers easily, plus — on a minority of groups
— a **bimodal datum offset** of about ±15 units that the available features simply do not disambiguate (a
fault-like subset jumps ~2x further). That last part is the irreducible bit. Swap this generator for your
real arrays to run the same diagnostics on your problem.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25, "font.size": 11})
rng = np.random.default_rng(0)

N_GROUPS, N_PER = 400, 60
DATUM = 15.0                      # half-gap between the two stratigraphic solutions
P_BIMODAL = 0.45                  # fraction of groups that are genuinely ambiguous

def make_dataset():
    truth, trend, groups, offset, is_bi = [], [], [], [], []
    for g in range(N_GROUPS):
        t = np.linspace(0, 1, N_PER)
        base = 8*np.sin(2.2*t + rng.uniform(0, 6)) + 4*t            # smooth per-group structural trend
        bi = rng.random() < P_BIMODAL
        mag = DATUM * (rng.uniform(1.5, 2.0) if rng.random() < 0.22 else 1.0)   # fault-like subset steps ~2x
        s = rng.choice([-1.0, 1.0]) * mag if bi else 0.0           # the unknowable datum offset
        y = base + s + rng.normal(0, 1.2, N_PER)                   # observed truth
        truth.append(y); trend.append(base); groups.append(np.full(N_PER, g))
        offset.append(s); is_bi.append(bi)
    return (np.concatenate(truth), np.concatenate(trend), np.concatenate(groups),
            np.array(offset), np.array(is_bi))

truth, trend, groups, offset, is_bi = make_dataset()
uniq = np.unique(groups)

# Candidate models: trend-only (predicts the MIDPOINT) and two that COMMIT to a datum sign.
pred_mid   = trend.copy()
pred_plus  = trend + DATUM
pred_minus = trend - DATUM
def rmse(a, b): return float(np.sqrt(np.mean((a-b)**2)))
for name, p in {"trend/midpoint": pred_mid, "commit +datum": pred_plus, "commit -datum": pred_minus}.items():
    print(f"{name:16s} pooled RMSE = {rmse(p, truth):.2f}")


## The picture in one figure: two equally-good answers

Before the tests, here is the whole problem in one plot. On an ambiguous well the horizontal log matches the
type well at **two** stratigraphic levels about `2*DATUM` apart. Both fit the known prefix about equally well,
so a model cannot tell which is right — and the choice shifts the *entire* lateral by a near-constant offset.


In [ ]:
gi = uniq[np.where(is_bi)[0][3]]              # one ambiguous example group
m = groups == gi
x = np.linspace(0, 1, m.sum())
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(x, truth[m], color="k", lw=2.4, label="true TVT (the answer)")
ax.plot(x, pred_plus[m],  color="#ee7722", lw=1.8, ls="--", label="commit to +datum")
ax.plot(x, pred_minus[m], color="#cc2222", lw=1.8, ls="--", label="commit to -datum")
ax.plot(x, pred_mid[m],   color="#2266cc", lw=2.0, label="midpoint (mean of the two)")
xa = 0.5
ax.annotate("", xy=(xa, pred_plus[m][30]), xytext=(xa, pred_minus[m][30]),
            arrowprops=dict(arrowstyle="<->", color="#555", lw=1.6))
ax.text(xa+0.015, pred_mid[m][30], f"  ≈ 2×{DATUM:.0f} ft\n  datum gap", va="center", color="#555")
ax.set_xlabel("position along lateral (normalised)"); ax.set_ylabel("TVT (ft)")
ax.set_title("One ambiguous well — two equally-plausible solutions, ~30 ft apart")
ax.legend(loc="upper left", fontsize=9); plt.tight_layout(); plt.show()


## Test 1 — Where does the error actually live? (tail concentration)

Pooled RMSE hides *who* is wrong. Compute each group's error, sort worst-first, and look at the **cumulative
share of total squared error**. If a small fraction of groups holds most of the error mass, the problem is
**tail-dominated**: polishing the easy majority barely moves the leaderboard, and the whole fight is in that
tail. (A near-diagonal curve instead means error is spread out and broad model gains still pay.)


In [ ]:
grp_rmse = np.array([rmse(pred_mid[groups==g], truth[groups==g]) for g in uniq])
grp_sse  = np.array([np.sum((pred_mid[groups==g]-truth[groups==g])**2) for g in uniq])
order = np.argsort(grp_sse)[::-1]
cum = np.cumsum(grp_sse[order]) / grp_sse.sum()
frac = np.arange(1, len(cum)+1) / len(cum)
top = int(0.10*len(cum))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
ax[0].fill_between(np.arange(len(grp_rmse)), np.sort(grp_rmse)[::-1], color="#cc7755", alpha=.8)
ax[0].axhline(np.median(grp_rmse), color="#114488", lw=1.5, ls="--",
              label=f"median well = {np.median(grp_rmse):.1f} ft")
ax[0].set_xlabel("wells, sorted worst -> best"); ax[0].set_ylabel("per-well RMSE (ft)")
ax[0].set_title("Per-well error is hugely uneven"); ax[0].legend()
ax[1].plot(frac*100, cum*100, lw=2.4, color="#cc2222")
ax[1].plot([0, 100], [0, 100], "k--", lw=1, alpha=.5, label="if error were uniform")
ax[1].scatter([10], [cum[top-1]*100], color="#cc2222", zorder=5)
ax[1].text(13, cum[top-1]*100-4, f"worst 10% of wells\n= {cum[top-1]*100:.0f}% of all error", va="center")
ax[1].set_xlabel("worst X% of wells"); ax[1].set_ylabel("cumulative % of squared error")
ax[1].set_title("Test 1: tail concentration"); ax[1].legend(loc="lower right")
plt.tight_layout(); plt.show()
print(f"median per-well RMSE = {np.median(grp_rmse):.2f} ft  |  worst 10% carry {cum[top-1]*100:.0f}% of squared error")


**Read it:** a curve that shoots up fast = a few groups dominate. That is the ROGII signature — a tail of
wells carries most of the error while the median well is already near-solved. The lesson: **track your median
per-group error, not just pooled RMSE**, and aim every remaining hour at the tail (and first check it is even
fixable — Tests 2 and 3).


## Test 2 — Is the headroom *reachable*? (oracle vs realizable)

The trap that wastes the most time. Suppose you have several candidate predictions per group. The **oracle**
that picks the best candidate per group looks amazing — huge headroom! But you can only ship a selector that
uses **information you actually have** (no peeking at the truth). If the thing that decides the best candidate
is **not in your features**, the selector is a coin-flip and the headroom is a **mirage** — chasing it adds
variance, not skill.


In [ ]:
cand = np.stack([pred_mid, pred_plus, pred_minus], 1)   # midpoint + the two datum commitments
oracle = pred_mid.copy(); guess = pred_mid.copy()
for g in uniq:
    mm = groups == g
    e = [np.mean((cand[mm, k]-truth[mm])**2) for k in range(3)]
    oracle[mm] = cand[mm, int(np.argmin(e))]           # ORACLE: peeks at truth -> best candidate (lower bound)
    guess[mm]  = cand[mm, rng.integers(3)]             # truth-free: can't tell the datum sign -> a blind guess

vals = {"oracle\n(cheats)": rmse(oracle, truth), "guess the\ndatum": rmse(guess, truth),
        "midpoint\n(realizable)": rmse(pred_mid, truth)}
fig, ax = plt.subplots(figsize=(6.8, 4.6))
bars = ax.bar(vals.keys(), vals.values(), color=["#22aa55", "#dd8800", "#2266cc"], width=.62)
for b, v in zip(bars, vals.values()): ax.text(b.get_x()+b.get_width()/2, v+0.25, f"{v:.1f}", ha="center")
ax.set_ylim(0, max(vals.values())*1.32)
ax.axhline(vals["midpoint\n(realizable)"], color="#2266cc", ls=":", alpha=.7)
ax.text(0, 6.0, "oracle looks like\nhuge headroom", ha="center", color="#117733", fontsize=10)
ax.text(1, 7.5, "but GUESSING the datum\nis worse, not better", ha="center", color="#aa5500", fontsize=10)
ax.text(2, vals["midpoint\n(realizable)"]+0.7, "best you can realize\nwithout the truth",
        ha="center", color="#1144aa", fontsize=9)
ax.set_ylabel("pooled RMSE (ft)"); ax.set_title("Test 2: oracle 'headroom' vs what you can realize")
plt.tight_layout(); plt.show()
print(f"oracle={rmse(oracle,truth):.1f}  guess={rmse(guess,truth):.1f}  midpoint={rmse(pred_mid,truth):.1f}"
      "  -> the oracle gap is a MIRAGE (guessing the datum is worse than the midpoint)")


**Read it:** the oracle bar is far lower — tempting. But you cannot reach it without the truth: a guesser
that *acts* on the oracle by picking a datum is **worse** than simply predicting the midpoint, because nothing
in the features decides the sign. That is a *non-realizable* gap — a classic data-limited signature. Always
pair an oracle estimate with a **CV-honest, truth-free selector** before you believe the headroom is real.


## Test 3 — When the target is bimodal, the *mean* is optimal

The deepest reason ROGII has a wall. Left: the distribution of each well's true datum offset — a big spike at
0 (the easy, solved wells) plus **two side-lobes at ±datum** (the ambiguous tail), with fault-like wells
further out. Right: on a genuinely ~50/50 well, expected RMSE as you push toward a mode is a **U-shape whose
minimum is at zero shift = the mean**. Committing is provably worse than the midpoint.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
ax[0].hist(offset, bins=np.linspace(-35, 35, 41), color="#5544aa", alpha=.85)
ax[0].axvline(-DATUM, color="#cc2222", ls="--"); ax[0].axvline(DATUM, color="#cc2222", ls="--")
ax[0].text(DATUM, ax[0].get_ylim()[1]*.6, f" +{DATUM:.0f}", color="#cc2222")
ax[0].text(-DATUM, ax[0].get_ylim()[1]*.6, f"-{DATUM:.0f} ", color="#cc2222", ha="right")
ax[0].set_xlabel("per-well datum offset (ft)"); ax[0].set_ylabel("number of wells")
ax[0].set_title("Bimodal datum: 0-spike (solved) + ±datum lobes (ambiguous)")

shift = np.linspace(0, DATUM, 60)                  # 0 = midpoint, DATUM = full commit to one mode
exp = np.sqrt(0.5*(DATUM-shift)**2 + 0.5*(DATUM+shift)**2)   # closed form, 50/50
ax[1].plot(shift, exp, lw=2.6, color="#7733aa")
ax[1].scatter([0], [exp[0]], color="#22aa55", zorder=5, s=60, label=f"midpoint = {exp[0]:.1f}")
ax[1].scatter([DATUM], [exp[-1]], color="#cc2222", zorder=5, s=60, label=f"full commit = {exp[-1]:.1f}")
ax[1].set_xlabel("shift toward one mode (ft)"); ax[1].set_ylabel("expected RMSE (ft)")
ax[1].set_title("Under 50/50 ambiguity, the mean minimizes RMSE"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"midpoint RMSE = {exp[0]:.1f}   full-commit RMSE = {exp[-1]:.1f}  (committing is WORSE)")


**Read it:** committing to a mode earns 0 ft half the time and `2*DATUM` the other half — its expected RMSE
(~21 ft) is **higher** than the flat midpoint (~15 ft). So a smooth, averaged prediction on ambiguous wells is
not lazy; it is **mathematically optimal**. You only beat it by *reducing the uncertainty below 50/50* — which
needs a real discriminating feature, not a cleverer guesser.


## The decision: data-limited or model-limited?

| Signal | Model-limited (keep building) | Data-limited (bank & stop) |
|---|---|---|
| **Test 1** tail | error spread out; broad gains pay | a few groups dominate; the tail is the whole fight |
| **Test 2** oracle | truth-free selector recovers much of the oracle | selector ≈ your blend ≫ oracle → mirage |
| **Test 3** bimodality | unimodal residuals; a better point estimate helps | ~50/50 residuals; the mean already minimizes RMSE |

If you see the right-hand column on all three — as ROGII's geosteering target does — the ceiling is **a
property of the data and the loss**, not a gap in your modeling. The correct move is to **bank your robust
solution** and spend remaining effort on diversity / robustness, not on chasing a non-realizable tail.

That is a genuinely useful thing to know early. For the geology that *creates* this ceiling (Milankovitch
cyclicity and the ±15 ft datum), see: https://www.kaggle.com/code/souldrive/decoding-eagle-ford-why-some-wells-are-hard

If these three tests save you a few days of chasing a mirage, an upvote is appreciated — and I would love to
hear which test was most useful on your own problem.


---
**More in this series (same competition):**

- [Decoding Eagle Ford: Why Some Wells Are Hard](https://www.kaggle.com/code/souldrive/decoding-eagle-ford-why-some-wells-are-hard) — the geology (Milankovitch cyclicity) behind the ±15 ft datum.
- [The ±15 ft datum: why honest CV bottoms out](https://www.kaggle.com/code/souldrive/the-15-ft-datum-why-honest-cv-bottoms-out) — the midpoint-optimality math on real wells, plus a multi-hypothesis (CNN-MTP) test.
